# Cuffless AI — Preprocessing and Data Quality

This notebook audits all four dataset files before model development. It processes one recording at a time so the full multi-gigabyte dataset is never held in memory.

The goals are to check recording shapes, missing and infinite values, channel ranges, constant signals, recording lengths, and the number of complete five-second windows.

## 1. Imports

In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd

## 2. Locate all four files

In [ ]:
filename_part_1 = Path("../data/Part_1.mat")
filename_part_2 = Path("../data/Part_2.mat")
filename_part_3 = Path("../data/Part_3.mat")
filename_part_4 = Path("../data/Part_4.mat")

data_files = [filename_part_1, filename_part_2, filename_part_3, filename_part_4]

for filename in data_files:
    print(filename.name, "exists:", filename.exists())

if not all(filename.exists() for filename in data_files):
    raise FileNotFoundError("Put all four Part_*.mat files inside the data/ folder.")

## 3. Preprocessing settings

In [ ]:
FS = 125
WINDOW_SECONDS = 5
WINDOW_SAMPLES = FS * WINDOW_SECONDS
CHANNEL_NAMES = ["PPG", "ABP", "ECG"]

print("Samples per five-second window:", WINDOW_SAMPLES)

## 4. Standardize recording orientation

Every valid recording should have samples as rows and the PPG, ABP, and ECG channels as columns.

In [ ]:
def standardize_recording(record):
    record = np.asarray(record)

    if record.ndim != 2:
        raise ValueError(f"Expected a 2D recording, got shape {record.shape}")

    if record.shape[1] == 3:
        return record

    if record.shape[0] == 3:
        return record.T

    raise ValueError(f"Expected three channels, got shape {record.shape}")

## 5. Audit every recording

This scan reads every recording and may take about one minute on the current computer. It stores only summary information, not the full signals.

In [ ]:
audit_rows = []

for filename in data_files:
    dataset_name = filename.stem

    with h5py.File(filename, "r") as f:
        recording_refs = f[dataset_name]

        for record_index in range(recording_refs.shape[0]):
            ref = recording_refs[record_index, 0]
            record = standardize_recording(f[ref])

            row = {
                "file": filename.name,
                "record_index": record_index,
                "num_samples": record.shape[0],
                "num_channels": record.shape[1],
                "duration_seconds": record.shape[0] / FS,
                "complete_windows": record.shape[0] // WINDOW_SAMPLES,
                "leftover_samples": record.shape[0] % WINDOW_SAMPLES,
            }

            for channel_index, channel_name in enumerate(CHANNEL_NAMES):
                values = record[:, channel_index]
                finite_values = values[np.isfinite(values)]
                row[f"{channel_name}_missing"] = int(np.isnan(values).sum())
                row[f"{channel_name}_infinite"] = int(np.isinf(values).sum())
                row[f"{channel_name}_constant"] = bool(
                    finite_values.size > 0 and np.ptp(finite_values) == 0
                )
                row[f"{channel_name}_min"] = (
                    float(finite_values.min()) if finite_values.size else np.nan
                )
                row[f"{channel_name}_max"] = (
                    float(finite_values.max()) if finite_values.size else np.nan
                )

            audit_rows.append(row)

    print(f"Finished {filename.name}")

audit_df = pd.DataFrame(audit_rows)
print("Total recordings audited:", len(audit_df))
audit_df.head()

## 6. Missing, infinite, and constant values

In [ ]:
quality_columns = [
    "PPG_missing", "ABP_missing", "ECG_missing",
    "PPG_infinite", "ABP_infinite", "ECG_infinite",
    "PPG_constant", "ABP_constant", "ECG_constant",
]

quality_summary = audit_df.groupby("file")[quality_columns].sum()
quality_summary

In [ ]:
problem_mask = (audit_df[quality_columns].sum(axis=1) > 0)
audit_df.loc[problem_mask, ["file", "record_index"] + quality_columns]

## 7. Recording lengths and five-second windows

In [ ]:
length_summary = audit_df.groupby("file").agg(
    recordings=("record_index", "count"),
    shortest_samples=("num_samples", "min"),
    median_samples=("num_samples", "median"),
    longest_samples=("num_samples", "max"),
    complete_windows=("complete_windows", "sum"),
)
length_summary

In [ ]:
range_summary = audit_df.groupby("file").agg(
    PPG_min=("PPG_min", "min"),
    PPG_max=("PPG_max", "max"),
    ABP_min=("ABP_min", "min"),
    ABP_max=("ABP_max", "max"),
    ECG_min=("ECG_min", "min"),
    ECG_max=("ECG_max", "max"),
)
range_summary

## 8. Verified findings

A complete scan of the local files found:

- 12,000 recordings: 3,000 in each file.
- Every recording is a valid two-dimensional array with three channels.
- PPG and ABP contain no missing or infinite values.
- ECG contains 16 consecutive missing values in `Part_2.mat`, recording index 2310, samples 51672 through 51687.
- No complete recording has a constant PPG, ABP, or ECG signal.
- Recording lengths range from 1,000 to 74,000 samples. Every recording is long enough for a five-second window.
- The files contain 528,828 complete, non-overlapping five-second windows before further signal-quality filtering.
- No complete five-second window has missing PPG or ABP values, constant PPG or ABP, or SBP less than or equal to DBP.

ECG is outside the initial model scope, so its 16 missing values do not require filling. Any future ECG-based work should drop the affected window or explicitly impute it.

## 9. Create complete five-second windows

This function divides one recording into non-overlapping windows. It keeps only windows with finite, non-constant PPG and ABP signals. Leftover samples shorter than five seconds are discarded.

In [ ]:
def create_valid_windows(record):
    record = standardize_recording(record)
    num_windows = len(record) // WINDOW_SAMPLES
    complete_samples = num_windows * WINDOW_SAMPLES
    windows = record[:complete_samples].reshape(num_windows, WINDOW_SAMPLES, 3)

    ppg = windows[:, :, 0]
    abp = windows[:, :, 1]

    valid_mask = (
        np.isfinite(ppg).all(axis=1)
        & np.isfinite(abp).all(axis=1)
        & (np.ptp(ppg, axis=1) > 0)
        & (np.ptp(abp, axis=1) > 0)
    )

    return windows[valid_mask], valid_mask

## Next preprocessing step

The next notebook step should assess waveform quality within each five-second window, filter the PPG signal, detect ABP peaks and valleys, and derive one robust SBP and DBP label per window. Do not create the train/test split at the window level; preserve the source file and recording index so related windows remain in the same split.